# Benchmark: VAE vs DDPM Wall-Clock Sampling Time

Purpose: measure real wall-clock time to generate images with your VAE and DDPM, so the README table can show actual minutes/seconds instead of just "1 forward pass" vs "1000 forward passes".

**No training required. No CelebA dataset required.** This only times forward passes through the model architecture.

### Before running
- Notebook Settings -> Accelerator -> GPU (T4 x2 or P100)
- Notebook Settings -> Internet -> **ON** (needed to `git clone` your repo — no dataset input needed at all)

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/AhmedAbdAlkreem/vae-vs-ddpm-celeba.git
%cd vae-vs-ddpm-celeba
!pip install -q -r requirements.txt

## 2. Instantiate the models

Random weights are fine here — we're timing compute (architecture + resolution + batch size), not measuring output quality. If you still have a saved checkpoint from Kaggle output or downloaded locally, you can load it instead (see commented-out block), but it changes nothing about the timing.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from configs.config import Config
from src.models.vae import VAE
from src.models.unet import UNet

cfg = Config()

vae = VAE.from_config(cfg.model).cuda().eval()
unet = UNet.from_config(cfg.model).cuda().eval()

# --- OPTIONAL: load real trained checkpoints instead of random weights ---
# from src.utils.checkpoint import load_checkpoint
# load_checkpoint(vae, "path/to/vae_checkpoint.pt")
# load_checkpoint(unet, "path/to/ddpm_checkpoint.pt")

print("Models loaded on GPU.")

## 3. Benchmark helper

`torch.cuda.synchronize()` is mandatory here — without it, we'd time how fast Python launches CUDA kernels, not how long the GPU takes to actually finish them, which would make the numbers look artificially fast.

In [ ]:
import time

def benchmark_generation(generate_fn, n_images, batch_size, warmup_batches=2):
    n_batches = max(1, n_images // batch_size)

    # warm-up: first CUDA calls include kernel compilation / memory allocation overhead
    for _ in range(warmup_batches):
        generate_fn(batch_size)
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(n_batches):
        generate_fn(batch_size)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    images_done = n_batches * batch_size
    return elapsed, images_done

## 4. Benchmark the VAE (1 decoder forward pass per image)

In [ ]:
def vae_generate(batch_size):
    with torch.no_grad():
        z = torch.randn(batch_size, cfg.model.latent_dim, device="cuda")
        return vae.decode(z)

vae_batch_size = 64
t_vae, n_vae = benchmark_generation(vae_generate, n_images=1000, batch_size=vae_batch_size)
print(f"VAE: {n_vae} images in {t_vae:.2f} sec  ->  {t_vae/60:.2f} min per 1000 images (batch={vae_batch_size})")

## 5. Benchmark the DDPM (1000 sequential U-Net passes per image)

Replace `full_ddpm_sample` below with your actual sampler function from `src/sampling/ddpm_sampler.py` — use the real one (not a bare `unet()` loop) so timing includes the noise-scheduling math your real `sample.py` does.

In [ ]:
from src.sampling.ddpm_sampler import full_ddpm_sample  # adjust import to match your actual function name

def ddpm_generate(batch_size):
    with torch.no_grad():
        return full_ddpm_sample(unet, batch_size=batch_size, n_steps=1000, cfg=cfg)

ddpm_batch_size = 32
t_ddpm, n_ddpm = benchmark_generation(ddpm_generate, n_images=1000, batch_size=ddpm_batch_size)
print(f"DDPM (full, 1000 steps): {n_ddpm} images in {t_ddpm:.2f} sec  ->  {t_ddpm/60:.2f} min per 1000 images (batch={ddpm_batch_size})")

## 6. (Optional) Benchmark DDIM (50 steps) too — closes out your README's open roadmap item

In [ ]:
from src.sampling.ddim_sampler import ddim_sample  # adjust import to match your actual function name

def ddim_generate(batch_size):
    with torch.no_grad():
        return ddim_sample(unet, batch_size=batch_size, n_steps=50, cfg=cfg)

ddim_batch_size = 32
t_ddim, n_ddim = benchmark_generation(ddim_generate, n_images=1000, batch_size=ddim_batch_size)
print(f"DDIM (50 steps): {n_ddim} images in {t_ddim:.2f} sec  ->  {t_ddim/60:.2f} min per 1000 images (batch={ddim_batch_size})")

## 7. Summary table — copy these numbers into your README

In [ ]:
import torch as _torch
gpu_name = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "CPU"

print(f"GPU: {gpu_name}\n")
print(f"{'Model':<20}{'Wall-clock / 1000 images':<30}{'Batch size':<12}")
print(f"{'VAE':<20}{f'{t_vae/60:.2f} min':<30}{vae_batch_size:<12}")
print(f"{'DDPM (full)':<20}{f'{t_ddpm/60:.2f} min':<30}{ddpm_batch_size:<12}")
try:
    print(f"{'DDIM (50 steps)':<20}{f'{t_ddim/60:.2f} min':<30}{ddim_batch_size:<12}")
except NameError:
    pass